# Model 7: FinBERT + PEFT (QLoRA vs LoRA vs Prefix Tuning)

**Model 7 of 7** in the NLP Sentiment Benchmark (Part A).

This notebook is **self-contained** — upload it to Kaggle as-is, add a GPU accelerator (T4 x1 or x2), and run all cells top to bottom. No other files needed.

At the end it saves the trained model + tokenizer + MLflow logs to `/kaggle/working/` — download those before your Kaggle session expires.

**This is the BEST model** — it feeds into the RL trading agent in Part B. We fine-tune `ProsusAI/finbert` (109M params) three different parameter-efficient ways and compare them like a research paper's ablation table. The winner (by val F1) gets registered in MLflow's Model Registry as `IndianFinSentiment`, stage=`Production`.

**Enable a GPU** (Settings -> Accelerator -> GPU T4 x1) before running — 4-bit quantization needs CUDA.

In [1]:
# transformers/peft/bitsandbytes/accelerate are usually pre-installed on Kaggle GPU images,
# but pin recent versions so QLoRA + SEQ_CLS PEFT works correctly.
# torchao is upgraded explicitly too: Kaggle images ship an old torchao (e.g. 0.10.0), and even
# though this notebook never uses torchao directly, PEFT's get_peft_model() still probes for it
# as one of several optional backends — if it finds a torchao version below what it requires, it
# raises ImportError instead of just skipping (that only happens when torchao isn't installed at
# all), which breaks LoRA/QLoRA/Prefix Tuning setup here even though we never touch torchao.
!pip install -q -U transformers peft bitsandbytes accelerate datasets mlflow evaluate "torchao>=0.16.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 96.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 94.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 88.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━

In [8]:
import os
import time
import pickle
import logging
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt   # uses Jupyter's inline backend on Kaggle — plots render in the notebook
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import (
    LoraConfig, PrefixTuningConfig, TaskType,
    get_peft_model, prepare_model_for_kbit_training,
)
import mlflow
import mlflow.transformers
from mlflow.tracking import MlflowClient

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

In [9]:
# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
LABEL2ID = {"positive": 0, "neutral": 1, "negative": 2}
ID2LABEL = {0: "positive", 1: "neutral", 2: "negative"}
NUM_LABELS = 3

MAX_LENGTH = 128
SEED = 42
BATCH_SIZE = 32
NUM_EPOCHS = 15          # raised from 6 — QLoRA's val F1 was still climbing at epoch 6 in the last
                         # run (never plateaued), so it likely has room left. Early stopping (below)
                         # protects methods like Prefix Tuning that converge/overfit much faster.
LEARNING_RATE = 2e-4     # per spec — higher LR is standard for adapter-only fine-tuning
PATIENCE = 3

TEST_SIZE = 0.15
VAL_SIZE = 0.10

FINBERT_MODEL = "ProsusAI/finbert"
MLFLOW_EXPERIMENT = "IndianFinSentiment"
MLFLOW_MODEL_NAME = "IndianFinSentiment"

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type != "cuda":
    print("WARNING: no GPU detected. 4-bit QLoRA requires a CUDA GPU — enable one in Kaggle settings.")

BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
RESULTS_DIR = os.path.join(BASE_DIR, "results")
MODELS_DIR = os.path.join(BASE_DIR, "saved_models")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# MLflow 3.x defaults to a SQLite-backed store (metrics/params live INSIDE
# mlflow.db, not as flat files) unless it finds existing file-store data.
# We pin the db to a known path — also required for the Model Registry step later.
mlflow.set_tracking_uri(f"sqlite:///{os.path.join(BASE_DIR, 'mlflow.db')}")

Using device: cuda


## 1. Load data

Same SentiFin + Indian_Financial_News combo used by Models 1-6, so results are directly comparable.

In [10]:
# ══════════════════════════════════════════════════════════════
# DATA LOADING
# Primary  : SentiFin (10,572 Indian stock market headlines)
# Secondary: kdave/Indian_Financial_News (26k rows, ET/Moneycontrol/BS)
# Fallback : financial_phrasebank (used only if SentiFin fails to load)
# ══════════════════════════════════════════════════════════════

def load_sentifin() -> pd.DataFrame:
    """
    Try SentiFin first. NOTE: as of writing, "Furthering/SentiFin" is not
    resolvable on the HF Hub (renamed/removed upstream) — this call is kept
    so it picks itself back up automatically if/when the repo reappears.
    Falls back to financial_phrasebank if unavailable, which is the normal
    path right now.
    """
    log.info("Loading SentiFin dataset...")
    try:
        ds = load_dataset("Furthering/SentiFin")
        df = pd.DataFrame(ds["train"])
        df = df.rename(columns={"sentence": "text"})
        df["label"] = df["label"].str.lower().str.strip()
        df = df[df["label"].isin(LABEL2ID.keys())]
        df["label_id"] = df["label"].map(LABEL2ID)
        df["source"] = "sentifin"
        log.info(f"  SentiFin loaded: {len(df)} samples")
        return df[["text", "label", "label_id", "source"]]
    except Exception as e:
        log.warning(f"  SentiFin unavailable ({e}). Loading financial_phrasebank...")
        return load_phrasebank()


def load_phrasebank() -> pd.DataFrame:
    """
    financial_phrasebank (sentences_50agree) — 4,846 English financial sentences.
    The original "financial_phrasebank" HF repo is script-based and no longer
    loadable by recent `datasets` versions, so we use a parquet mirror with
    identical content/labels instead.
    """
    log.info("Loading financial_phrasebank (via parquet mirror)...")
    ds = load_dataset("warwickai/financial_phrasebank_mirror")
    df = pd.DataFrame(ds["train"])
    df = df.rename(columns={"sentence": "text"})
    # mirror's ClassLabel is [negative, neutral, positive] -> ids 0,1,2
    # we use positive=0, neutral=1, negative=2, so remap accordingly
    remap = {0: 2, 1: 1, 2: 0}
    df["label_id"] = df["label"].map(remap)
    df["label"] = df["label_id"].map(ID2LABEL)
    df["source"] = "phrasebank"
    log.info(f"  financial_phrasebank loaded: {len(df)} samples")
    return df[["text", "label", "label_id", "source"]]


def load_indian_financial_news() -> Optional[pd.DataFrame]:
    """kdave/Indian_Financial_News — 26k rows from ET/Moneycontrol/Business Standard."""
    log.info("Loading Indian_Financial_News dataset...")
    try:
        ds = load_dataset("kdave/Indian_Financial_News")
        df = pd.DataFrame(ds["train"])

        text_col = next((c for c in ["Summary", "summary", "title", "Title", "text", "Text"] if c in df.columns), None)
        label_col = next((c for c in ["Sentiment Analysis", "sentiment", "Sentiment", "label"] if c in df.columns), None)
        if text_col is None or label_col is None:
            log.warning("  Could not find text/label column. Skipping.")
            return None

        df = df[[text_col, label_col]].dropna()
        df.columns = ["text", "label_raw"]

        label_map = {}
        for k in ["positive", "Positive", "POSITIVE"]:
            label_map[k] = "positive"
        for k in ["neutral", "Neutral", "NEUTRAL"]:
            label_map[k] = "neutral"
        for k in ["negative", "Negative", "NEGATIVE"]:
            label_map[k] = "negative"

        df["label"] = df["label_raw"].map(label_map)
        df = df.dropna(subset=["label"])
        df["label_id"] = df["label"].map(LABEL2ID)
        df["source"] = "indian_news"
        log.info(f"  Indian_Financial_News loaded: {len(df)} samples")
        return df[["text", "label", "label_id", "source"]]
    except Exception as e:
        log.warning(f"  Indian_Financial_News unavailable: {e}")
        return None


def load_and_combine() -> pd.DataFrame:
    """Load all sources, merge, clean, dedupe. Returns one clean DataFrame."""
    dfs = [load_sentifin()]
    secondary = load_indian_financial_news()
    if secondary is not None:
        dfs.append(secondary)

    df = pd.concat(dfs, ignore_index=True)

    df = df.dropna(subset=["text", "label_id"])
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"].str.len() > 10]         # drop near-empty rows
    df = df[df["text"].str.len() < 1000]       # drop unusually long rows
    df = df.drop_duplicates(subset=["text"])

    log.info(f"Combined dataset: {len(df)} samples")
    log.info(f"Sources: {df['source'].value_counts().to_dict()}")
    for label, count in df["label"].value_counts().items():
        log.info(f"  {label:>10}: {count:>6} ({count/len(df)*100:.1f}%)")

    return df.reset_index(drop=True)


def split_data(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Stratified train/val/test split so class balance holds in every split."""
    train_val_df, test_df = train_test_split(
        df, test_size=TEST_SIZE, stratify=df["label_id"], random_state=SEED
    )
    train_df, val_df = train_test_split(
        train_val_df, test_size=VAL_SIZE / (1 - TEST_SIZE),
        stratify=train_val_df["label_id"], random_state=SEED
    )
    log.info(f"Split sizes -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


df_all = load_and_combine()
train_df, val_df, test_df = split_data(df_all)

2026-07-17 10:00:16,070 | INFO | Loading SentiFin dataset...
2026-07-17 10:00:16,195 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/Furthering/SentiFin/resolve/main/README.md "HTTP/1.1 401 Unauthorized"
2026-07-17 10:00:16,196 | WARNING |   SentiFin unavailable (Dataset 'Furthering/SentiFin' doesn't exist on the Hub or cannot be accessed.). Loading financial_phrasebank...
2026-07-17 10:00:16,197 | INFO | Loading financial_phrasebank (via parquet mirror)...
2026-07-17 10:00:16,298 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/warwickai/financial_phrasebank_mirror/resolve/main/README.md "HTTP/1.1 404 Not Found"
2026-07-17 10:00:16,379 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/warwickai/financial_phrasebank_mirror "HTTP/1.1 200 OK"
2026-07-17 10:00:16,467 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/warwickai/financial_phrasebank_mirror/resolve/ff5746068c6861a5cb980aa178d80d65161c6376/financial_phrasebank_mirror.py "HTTP/1.1 4

## 2. Tokenize with FinBERT's own (BERT WordPiece) tokenizer

Unlike Models 1-6, we do NOT build a custom tokenizer here — FinBERT was pretrained with a specific WordPiece vocabulary, so we must reuse its tokenizer.

In [13]:
hf_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)

def tokenize_df(df: pd.DataFrame):
    enc = hf_tokenizer(
        df["text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    labels = torch.tensor(df["label_id"].tolist(), dtype=torch.long)
    return TensorDataset(enc["input_ids"], enc["attention_mask"], labels)


train_ds = tokenize_df(train_df)
val_ds = tokenize_df(val_df)
test_ds = tokenize_df(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# class weights for the imbalanced 3-class problem
class_counts = train_df["label_id"].value_counts().sort_index().values
class_weights = torch.tensor(len(train_df) / (NUM_LABELS * class_counts), dtype=torch.float32).to(DEVICE)
print(f"Class weights (positive, neutral, negative): {class_weights.tolist()}")

2026-07-17 10:02:21,406 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 10:02:21,417 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-17 10:02:21,503 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 10:02:21,514 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-17 10:02:21,526 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

2026-07-17 10:02:21,621 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-07-17 10:02:21,704 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-07-17 10:02:21,787 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/vocab.txt "HTTP/1.1 307 Temporary Redirect"
2026-07-17 10:02:21,798 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/vocab.txt "HTTP/1.1 200 OK"
2026-07-17 10:02:21,810 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/vocab.txt "HTTP/1.1 200 OK"


vocab.txt: 0.00B [00:00, ?B/s]

2026-07-17 10:02:21,913 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/tokenizer.json "HTTP/1.1 404 Not Found"
2026-07-17 10:02:21,995 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
2026-07-17 10:02:22,095 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 10:02:22,106 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/special_tokens_map.json "HTTP/1.1 200 OK"
2026-07-17 10:02:22,118 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

2026-07-17 10:02:22,213 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Class weights (positive, neutral, negative): [1.1121289730072021, 0.8237215876579285, 1.1276229619979858]


## 3. Shared train/eval helpers

In [11]:
def compute_metrics(y_true, y_pred) -> dict:
    per_class = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2])
    return {
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "f1_weighted": round(f1_score(y_true, y_pred, average="weighted"), 4),
        "f1_positive": round(per_class[0], 4),
        "f1_neutral": round(per_class[1], 4),
        "f1_negative": round(per_class[2], 4),
    }


def save_confusion_matrix(y_true, y_pred, model_name: str) -> str:
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm_pct, annot=True, fmt=".2%", cmap="Blues",
                xticklabels=["positive", "neutral", "negative"],
                yticklabels=["positive", "neutral", "negative"], ax=ax, linewidths=0.5)
    ax.set_title(f"Confusion Matrix - {model_name}", fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    path = os.path.join(RESULTS_DIR, f"cm_{model_name.lower().replace(' ', '_').replace('/', '-')}.png")
    plt.tight_layout()
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.close()
    return path


def trainable_param_stats(model) -> dict:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return {"trainable_params": trainable, "total_params": total, "trainable_pct": round(100 * trainable / total, 4)}


def train_peft_epoch(model, loader, optimizer, criterion, scheduler=None) -> float:
    model.train()
    total_loss = 0.0
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item() * input_ids.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_peft(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        loss = criterion(logits, labels)
        total_loss += loss.item() * input_ids.size(0)

        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, compute_metrics(all_labels, all_preds), all_preds, all_labels


def fit_peft(model, model_name: str, epochs=NUM_EPOCHS, lr=LEARNING_RATE, patience=PATIENCE):
    """Same early-stopping-on-val-F1 pattern used for Models 3-6, adapted to HF's forward() signature.

    Uses a linear warmup + linear decay learning rate schedule (10% warmup) — standard for
    transformer fine-tuning, and it directly addresses what we saw in the last run: Prefix Tuning's
    val F1 peaked at epoch 1 and got steadily worse afterward under a flat learning rate, which is
    a classic sign the LR stayed too high for too long. Decaying it over training gives every
    variant a chance to settle into a better minimum instead of overshooting it.

    NOTE: the best-epoch checkpoint only snapshots/restores the TRAINABLE parameters (the
    LoRA/prefix adapter weights + classifier head), not the full model.state_dict(). For QLoRA,
    the frozen base model is 4-bit quantized via bitsandbytes, which stores extra per-weight
    buffers (absmax, quant_map, nested_absmax, ...) alongside each quantized tensor — those don't
    round-trip cleanly through state_dict()/load_state_dict() (cloning + reloading them raises
    "Unexpected key(s) in state_dict"). Since the base model is frozen and never changes during
    training, only the trainable adapter parameters ever need to be checkpointed anyway.
    """
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    total_steps = len(train_loader) * epochs
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    best_val_f1, best_state, epochs_no_improve = -1.0, None, 0
    start = time.time()

    for epoch in range(1, epochs + 1):
        train_loss = train_peft_epoch(model, train_loader, optimizer, criterion, scheduler)
        val_loss, val_metrics, _, _ = evaluate_peft(model, val_loader, criterion)

        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        for k, v in val_metrics.items():
            mlflow.log_metric(f"val_{k}", v, step=epoch)

        print(f"[{model_name}] Epoch {epoch}/{epochs} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | val_f1={val_metrics['f1_weighted']:.4f}")

        if val_metrics["f1_weighted"] > best_val_f1:
            best_val_f1 = val_metrics["f1_weighted"]
            best_state = {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    train_time = time.time() - start
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in best_state:
                param.copy_(best_state[name].to(param.device))
    return model, train_time, best_val_f1

## 4. Variant A — QLoRA (4-bit NF4 quantized base + LoRA r=8)

In [30]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    # Without this, the classifier head (freshly initialized for our 3-class problem,
    # not part of FinBERT's pretrained weights) also gets quantized into a non-floating-point
    # Linear4bit layer. PEFT then can't set requires_grad=True on it, and get_peft_model()
    # crashes with "only Tensors of floating point dtype can require gradients".
    llm_int8_skip_modules=["classifier"],
)

base_model_qlora = AutoModelForSequenceClassification.from_pretrained(
    FINBERT_MODEL, num_labels=NUM_LABELS, quantization_config=bnb_config, device_map={"": 0}
)
base_model_qlora = prepare_model_for_kbit_training(base_model_qlora)

qlora_config = LoraConfig(
    r=8, lora_alpha=16, target_modules=["query", "value"],
    lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_CLS,
)
qlora_model = get_peft_model(base_model_qlora, qlora_config)
qlora_model.print_trainable_parameters()

mlflow.set_experiment(MLFLOW_EXPERIMENT)
with mlflow.start_run(run_name="FinBERT-QLoRA"):
    mlflow.log_params({
        "peft_method": "QLoRA", "quant": "4bit-nf4", "r": 8, "lora_alpha": 16,
        "target_modules": "query,value", "lora_dropout": 0.1, "lr": LEARNING_RATE,
    })

    qlora_model, qlora_time, qlora_val_f1 = fit_peft(qlora_model, "QLoRA")

    test_loss, test_metrics, test_preds, test_labels = evaluate_peft(
        qlora_model, test_loader, nn.CrossEntropyLoss(weight=class_weights)
    )
    stats = trainable_param_stats(qlora_model)

    for k, v in test_metrics.items():
        mlflow.log_metric(f"test_{k}", v)
    mlflow.log_metric("train_time_seconds", round(qlora_time, 2))
    mlflow.log_metrics(stats)

    cm_path = save_confusion_matrix(test_labels, test_preds, "FinBERT-QLoRA")
    mlflow.log_artifact(cm_path)

    qlora_model.save_pretrained(os.path.join(MODELS_DIR, "finbert_qlora_adapter"))
    mlflow.transformers.log_model(
        transformers_model={"model": qlora_model, "tokenizer": hf_tokenizer},
        name="model", task="text-classification",
    )
    qlora_run_id = mlflow.active_run().info.run_id

qlora_results = {"name": "QLoRA (r=8, q+v)", "test_metrics": test_metrics, "stats": stats,
                  "train_time": qlora_time, "run_id": qlora_run_id}
print(qlora_results)

2026-07-16 21:10:44,235 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 21:10:44,252 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-16 21:10:44,318 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 21:10:44,334 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-16 21:10:44,405 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-07-16 21:10:44,518 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert "HTTP/1.1 200 OK"
2026-07-16 21:10:44,630 | INFO | HTTP Request: GET htt

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-07-16 21:10:45,004 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-07-16 21:10:45,076 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors "HTTP/1.1 302 Found"
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


trainable params: 297,219 || all params: 109,781,766 || trainable%: 0.2707
[QLoRA] Epoch 1/15 | train_loss=1.1038 | val_loss=0.6152 | val_f1=0.7101


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


...

[QLoRA] Epoch 8/15 | train_loss=0.3878 | val_loss=0.5666 | val_f1=0.7589


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


[QLoRA] Epoch 9/15 | train_loss=0.3684 | val_loss=0.5693 | val_f1=0.7602
Early stopping at epoch 9.


2026-07-16 21:30:56,931 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 21:30:56,947 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-16 21:30:57,010 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 21:30:57,026 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026/07/16 21:30:57 INFO mlflow.transformers: Overriding save_pretrained to False for PEFT models, following the Transformers behavior. The PEFT adaptor and config will be saved, but the base model weights will not and reference to the HuggingFace Hub repository will be logged instead.
2026-07-16 21:30:57,203 | INFO | HT

{'name': 'QLoRA (r=8, q+v)', 'test_metrics': {'accuracy': 0.7632, 'f1_weighted': 0.7593, 'f1_positive': np.float64(0.7725), 'f1_neutral': np.float64(0.6988), 'f1_negative': np.float64(0.8289)}, 'stats': {'trainable_params': 297219, 'total_params': 67019526, 'trainable_pct': 0.4435}, 'train_time': 1203.8805031776428, 'run_id': '2a1fd3c246dc45e1b784b52792334803'}


## 5. Variant B — plain LoRA (no quantization, r=16, q+k+v)

In [31]:
base_model_lora = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL, num_labels=NUM_LABELS).to(DEVICE)

lora_config = LoraConfig(
    r=16, lora_alpha=16, target_modules=["query", "key", "value"],
    lora_dropout=0.1, bias="none", task_type=TaskType.SEQ_CLS,
)
lora_model = get_peft_model(base_model_lora, lora_config)
lora_model.print_trainable_parameters()

with mlflow.start_run(run_name="FinBERT-LoRA"):
    mlflow.log_params({
        "peft_method": "LoRA", "quant": "none", "r": 16, "lora_alpha": 16,
        "target_modules": "query,key,value", "lora_dropout": 0.1, "lr": LEARNING_RATE,
    })

    lora_model, lora_time, lora_val_f1 = fit_peft(lora_model, "LoRA")

    test_loss, test_metrics, test_preds, test_labels = evaluate_peft(
        lora_model, test_loader, nn.CrossEntropyLoss(weight=class_weights)
    )
    stats = trainable_param_stats(lora_model)

    for k, v in test_metrics.items():
        mlflow.log_metric(f"test_{k}", v)
    mlflow.log_metric("train_time_seconds", round(lora_time, 2))
    mlflow.log_metrics(stats)

    cm_path = save_confusion_matrix(test_labels, test_preds, "FinBERT-LoRA")
    mlflow.log_artifact(cm_path)

    lora_model.save_pretrained(os.path.join(MODELS_DIR, "finbert_lora_adapter"))
    mlflow.transformers.log_model(
        transformers_model={"model": lora_model, "tokenizer": hf_tokenizer},
        name="model", task="text-classification",
    )
    lora_run_id = mlflow.active_run().info.run_id

lora_results = {"name": "LoRA (r=16, q+k+v)", "test_metrics": test_metrics, "stats": stats,
                 "train_time": lora_time, "run_id": lora_run_id}
print(lora_results)

2026-07-16 21:30:57,628 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 21:30:57,644 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-16 21:30:57,710 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 21:30:57,727 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-16 21:30:57,794 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-07-16 21:30:57,867 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert "HTTP/1.1 200 OK"
2026-07-16 21:30:57,966 | INFO | HTTP Request: GET htt

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-07-16 21:30:58,202 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/commits/refs%2Fpr%2F29 "HTTP/1.1 200 OK"
2026-07-16 21:30:58,270 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-07-16 21:30:58,341 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors "HTTP/1.1 302 Found"


trainable params: 887,043 || all params: 110,371,590 || trainable%: 0.8037
[LoRA] Epoch 1/15 | train_loss=1.1125 | val_loss=0.6075 | val_f1=0.7179
[LoRA] Epoch 2/15 | train_loss=0.5688 | val_loss=0.5560 | val_f1=0.7479
[LoRA] Epoch 3/15 | train_loss=0.5090 | val_loss=0.5410 | val_f1=0.7370
[LoRA] Epoch 4/15 | train_loss=0.4696 | val_loss=0.5237 | val_f1=0.7596
[LoRA] Epoch 5/15 | train_loss=0.4290 | val_loss=0.5281 | val_f1=0.7612
[LoRA] Epoch 6/15 | train_loss=0.3982 | val_loss=0.5441 | val_f1=0.7601
[LoRA] Epoch 7/15 | train_loss=0.3679 | val_loss=0.5539 | val_f1=0.7756
[LoRA] Epoch 8/15 | train_loss=0.3425 | val_loss=0.6011 | val_f1=0.7626
[LoRA] Epoch 9/15 | train_loss=0.3122 | val_loss=0.6184 | val_f1=0.7601
[LoRA] Epoch 10/15 | train_loss=0.2988 | val_loss=0.6269 | val_f1=0.7689
Early stopping at epoch 10.


2026-07-16 22:06:38,181 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 22:06:38,198 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-16 22:06:38,262 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-16 22:06:38,277 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026/07/16 22:06:38 INFO mlflow.transformers: Overriding save_pretrained to False for PEFT models, following the Transformers behavior. The PEFT adaptor and config will be saved, but the base model weights will not and reference to the HuggingFace Hub repository will be logged instead.
2026-07-16 22:06:38,452 | INFO | HT

{'name': 'LoRA (r=16, q+k+v)', 'test_metrics': {'accuracy': 0.7735, 'f1_weighted': 0.7715, 'f1_positive': np.float64(0.7722), 'f1_neutral': np.float64(0.7245), 'f1_negative': np.float64(0.8353)}, 'stats': {'trainable_params': 887043, 'total_params': 110371590, 'trainable_pct': 0.8037}, 'train_time': 2119.806328058243, 'run_id': 'd2c17dbdfd3e4957a1ba53a38e5d8f07'}


## 6. Variant C — Prefix Tuning (20 virtual tokens)

In [ ]:
base_model_prefix = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL, num_labels=NUM_LABELS).to(DEVICE)

prefix_config = PrefixTuningConfig(task_type=TaskType.SEQ_CLS, num_virtual_tokens=20)
prefix_model = get_peft_model(base_model_prefix, prefix_config)
prefix_model.print_trainable_parameters()

with mlflow.start_run(run_name="FinBERT-PrefixTuning"):
    mlflow.log_params({"peft_method": "Prefix Tuning", "num_virtual_tokens": 20, "lr": LEARNING_RATE})

    prefix_model, prefix_time, prefix_val_f1 = fit_peft(prefix_model, "PrefixTuning")

    test_loss, test_metrics, test_preds, test_labels = evaluate_peft(
        prefix_model, test_loader, nn.CrossEntropyLoss(weight=class_weights)
    )
    stats = trainable_param_stats(prefix_model)

    for k, v in test_metrics.items():
        mlflow.log_metric(f"test_{k}", v)
    mlflow.log_metric("train_time_seconds", round(prefix_time, 2))
    mlflow.log_metrics(stats)

    cm_path = save_confusion_matrix(test_labels, test_preds, "FinBERT-PrefixTuning")
    mlflow.log_artifact(cm_path)

    prefix_model.save_pretrained(os.path.join(MODELS_DIR, "finbert_prefix_adapter"))
    mlflow.transformers.log_model(
        transformers_model={"model": prefix_model, "tokenizer": hf_tokenizer},
        name="model", task="text-classification",
    )
    prefix_run_id = mlflow.active_run().info.run_id

prefix_results = {"name": "Prefix Tuning (20 tokens)", "test_metrics": test_metrics, "stats": stats,
                   "train_time": prefix_time, "run_id": prefix_run_id}
print(prefix_results)

2026-07-17 10:02:29,872 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 10:02:29,883 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-17 10:02:29,969 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-17 10:02:29,981 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ProsusAI/finbert/4556d13015211d73dccd3fdd39d39232506f3e43/config.json "HTTP/1.1 200 OK"
2026-07-17 10:02:30,063 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"
2026-07-17 10:02:30,173 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert "HTTP/1.1 200 OK"
2026-07-17 10:02:30,352 | INFO | HTTP Request: GET htt

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-07-17 10:02:30,619 | INFO | HTTP Request: GET https://huggingface.co/api/models/ProsusAI/finbert/commits/refs%2Fpr%2F29 "HTTP/1.1 200 OK"
2026-07-17 10:02:30,704 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors.index.json "HTTP/1.1 404 Not Found"
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
2026-07-17 10:02:30,789 | INFO | HTTP Request: HEAD https://huggingface.co/ProsusAI/finbert/resolve/refs%2Fpr%2F29/model.safetensors "HTTP/1.1 302 Found"


trainable params: 370,947 || all params: 109,855,494 || trainable%: 0.3377
[PrefixTuning] Epoch 1/15 | train_loss=1.1120 | val_loss=0.7293 | val_f1=0.6722
[PrefixTuning] Epoch 2/15 | train_loss=0.6933 | val_loss=0.6922 | val_f1=0.6889
[PrefixTuning] Epoch 3/15 | train_loss=0.6666 | val_loss=0.6663 | val_f1=0.6931
[PrefixTuning] Epoch 4/15 | train_loss=0.6438 | val_loss=0.6723 | val_f1=0.6911
[PrefixTuning] Epoch 5/15 | train_loss=0.6358 | val_loss=0.6428 | val_f1=0.7014


## 7. PEFT ablation comparison table (Table 1 style)

In [ ]:
all_results = [qlora_results, lora_results, prefix_results]

print(f"{'Method':<22} {'F1 (weighted)':>14} {'Accuracy':>10} {'Trainable %':>13} {'Time (s)':>10}")
print("-" * 75)
for r in all_results:
    m, s = r["test_metrics"], r["stats"]
    print(f"{r['name']:<22} {m['f1_weighted']:>14.4f} {m['accuracy']:>10.4f} "
          f"{s['trainable_pct']:>12.3f}% {r['train_time']:>10.1f}")

comparison_df = pd.DataFrame([
    {"Method": r["name"], "F1 Weighted": r["test_metrics"]["f1_weighted"],
     "Accuracy": r["test_metrics"]["accuracy"], "Trainable %": r["stats"]["trainable_pct"],
     "Train Time (s)": r["train_time"]}
    for r in all_results
])
comparison_df.to_csv(os.path.join(RESULTS_DIR, "peft_ablation_table.csv"), index=False)
comparison_df

## 8. Register the best variant in MLflow Model Registry (alias=Production)

In [ ]:
best = max(all_results, key=lambda r: r["test_metrics"]["f1_weighted"])
print(f"Best PEFT variant: {best['name']} (F1={best['test_metrics']['f1_weighted']:.4f})")

client = MlflowClient()
model_uri = f"runs:/{best['run_id']}/model"

try:
    registered = mlflow.register_model(model_uri=model_uri, name=MLFLOW_MODEL_NAME)
    # MLflow deprecated "stages" (transition_model_version_stage) in favor of aliases —
    # set_registered_model_alias is the current API for marking a version as the one to serve.
    client.set_registered_model_alias(name=MLFLOW_MODEL_NAME, alias="Production", version=registered.version)
    print(f"Registered '{MLFLOW_MODEL_NAME}' v{registered.version} -> alias=Production")
except Exception as e:
    print(f"Registry step skipped/failed ({e}). The winning adapter files are still saved locally below.")

hf_tokenizer.save_pretrained(os.path.join(MODELS_DIR, "finbert_tokenizer"))
print(f"\nWinning adapter to use in Part B (RL state builder): saved_models/finbert_{best['name'].split()[0].lower()}_adapter")
print(f"\nDownload these from the Kaggle notebook's Output tab before your session ends:")
print(f"  - {MODELS_DIR}/  (3 adapters + tokenizer)")
print(f"  - {RESULTS_DIR}/  (confusion matrices, peft_ablation_table.csv)")
print(f"  - {os.path.join(BASE_DIR, 'mlflow.db')}  (all MLflow metrics/params — NOT just the mlruns/ folder!)")
print(f"  - {os.path.join(BASE_DIR, 'mlruns')}/  (MLflow artifacts)")